# V-ChartText – Chạy baseline Mức 1 trên Kaggle

**Chuẩn bị (làm một lần):**
1. Tài khoản Kaggle đã xác minh số điện thoại (bắt buộc để bật GPU và Internet).
2. Tạo dataset riêng tư chứa ảnh: *Datasets → New Dataset*, tải lên file zip của thư mục **Ảnh biểu đồ** (giữ các thư mục con "cột", "đường", "tròn", "kết hợp", "miền"), chọn **Private**.
3. Trong notebook này: *Add Input* → chọn dataset vừa tạo.
4. *Settings → Accelerator → GPU T4 x2*; *Settings → Internet → On*.
5. *Add-ons → Secrets*: thêm `GH_TOKEN` (GitHub token, quyền *Contents: Read*) và bật cho notebook này.

Repo được clone vào `/tmp` (không lưu, để token và ảnh SGK không nằm trong output). Kết quả ghi thẳng vào `/kaggle/working/outputs`, nên được giữ lại khi *Save Version*.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
GH_USER = 'rangha26'          # repo cá nhân; đổi sang 'dangthihoangnganit6' khi chạy repo nhóm
BRANCH  = 'dev'
REPO    = '/tmp/V-ChartText'
OUT     = '/kaggle/working/outputs'
MODEL   = 'Qwen/Qwen3-VL-2B-Instruct'
IDS     = 'bar_001,line_001,pie_001,combo_001,area_001'

# Tự tìm thư mục ảnh trong /kaggle/input (thư mục có thư mục con tên chứa "cột")
import glob, os, unicodedata
nfc = lambda s: unicodedata.normalize('NFC', s).lower()
RAW = next(d for d in sorted(glob.glob('/kaggle/input/**/', recursive=True))
           if any('cột' in nfc(x) for x in os.listdir(d)))
print('RAW =', RAW)

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GH_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/{GH_USER}/V-ChartText.git {REPO}
%cd {REPO}
!git clean -fq -- data/labels data/manifest.csv   # bỏ nhãn khung sinh tự động (nếu có) để pull không bị chặn
!git pull -q origin {BRANCH}
!git log --oneline -1

# outputs/ của repo trỏ sang /kaggle/working/outputs để kết quả không mất
os.makedirs(OUT, exist_ok=True)
if not os.path.islink('outputs'):
    !rm -rf outputs && ln -s {OUT} outputs
!ls -la outputs

In [ ]:
!pip install -q "transformers>=4.57.0" accelerate timm einops sentencepiece

## 1. Nhập ảnh và kiểm tra nhãn

In [ ]:
!python src/prepare_dataset.py --raw "{RAW}"
!python eval/validate_labels.py | tail -8

## 2. Qwen3-VL-2B: chạy thử 5 ảnh
Muốn chạy lại từ đầu thì chạy cell chuyển kết quả cũ trước (script bỏ qua id đã có trong file output).

In [ ]:
# Chuyển kết quả cũ sang outputs_old/ (chỉ chạy khi muốn chạy lại)
!mkdir -p /kaggle/working/outputs_old && mv {OUT}/*.jsonl /kaggle/working/outputs_old/ 2>/dev/null; ls /kaggle/working/outputs_old

In [ ]:
# extract chạy trên GPU 0, describe trên GPU 1, song song
!CUDA_VISIBLE_DEVICES=0 python src/infer_qwen.py --model {MODEL} --task extract  --ids {IDS} > /kaggle/working/log_qwen_extract.txt  2>&1 & \
 CUDA_VISIBLE_DEVICES=1 python src/infer_qwen.py --model {MODEL} --task describe --ids {IDS} > /kaggle/working/log_qwen_describe.txt 2>&1 & wait
!tail -n 8 /kaggle/working/log_qwen_extract.txt /kaggle/working/log_qwen_describe.txt

In [ ]:
import json
def parse_json(s):
    s = s.strip()
    if s.startswith('```'):
        s = s.split('\n', 1)[1].rsplit('```', 1)[0]
    return json.loads(s)

def show(pattern='*'):
    # In đầy đủ output, kèm số token, dtype và kiểm tra JSON
    for f in sorted(glob.glob(f'outputs/{pattern}.jsonl')):
        print('=' * 20, f)
        for line in open(f, encoding='utf-8'):
            r = json.loads(line); out = r['output']
            n = r.get('n_new_tokens')
            info = f"{n} token" + ("  ⚠ CHẠM max_new_tokens" if n and n >= 760 else "") if n else ""
            info += f" | {r.get('dtype', '')}"
            if '__extract__' in f:
                try:
                    parse_json(out); info += " | JSON hợp lệ"
                except Exception as e:
                    info += f" | JSON LỖI: {e}"
            print(f"--- {r['id']} ({r['latency_s']}s, {info})\n{out}\n")

show('Qwen*')

## 3. Vintern-1B-v3.5
Nếu báo lỗi khi nạp mô hình (mã `trust_remote_code` chưa hợp với transformers mới): chạy cell *hạ phiên bản*, rồi **Run → Restart & clear cell outputs** và chạy lại từ cell cấu hình (bỏ qua cell cài đặt). Chạy hết Qwen trước khi hạ phiên bản.

`repetition_penalty` mặc định 1.5 theo model card. Nếu JSON của extract bị hỏng, chạy lại extract với `--repetition_penalty 1.05` (ghi vào báo cáo: đây là cấu hình giải mã, không phải thay đổi prompt).

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/infer_vintern.py --task extract  --ids {IDS} > /kaggle/working/log_vintern_extract.txt  2>&1 & \
 CUDA_VISIBLE_DEVICES=1 python src/infer_vintern.py --task describe --ids {IDS} > /kaggle/working/log_vintern_describe.txt 2>&1 & wait
!tail -n 8 /kaggle/working/log_vintern_extract.txt /kaggle/working/log_vintern_describe.txt
show('Vintern*')

In [ ]:
# Chỉ chạy khi cell trên lỗi nạp mô hình, sau đó Restart
!pip install -q "transformers<4.50"

## 4. Chấm điểm (chỉ tính các ảnh đã gán nhãn xong, `status` khác `todo`)

In [ ]:
for f in sorted(glob.glob('outputs/*__extract__*.jsonl')):
    !python eval/score_extract.py "{f}"
for f in sorted(glob.glob('outputs/*__describe__*.jsonl')):
    !python eval/check_describe.py "{f}"
!mkdir -p /kaggle/working/results && cp -r eval/results/* /kaggle/working/results/ 2>/dev/null; ls /kaggle/working/results

## 5. Chạy toàn bộ bộ dữ liệu
Bỏ `--ids`. Nên chạy bằng **Save Version → Save & Run All (Commit)**: Kaggle chạy nền tối đa 12 giờ, không cần mở trình duyệt, và giữ lại `/kaggle/working`. Trước khi commit, bỏ dấu `#` ở cell dưới và chỉ để lại các cell cần thiết (cấu hình, clone, cài đặt, nhập ảnh, cell này).

**Chạy tiếp khi bị ngắt:** *Add Input → Notebook Output* của phiên bản trước, rồi bỏ `#` ở dòng `cp` để chép các file `.jsonl` cũ vào `outputs/`; script tự bỏ qua ảnh đã chạy.

In [ ]:
# !cp /kaggle/input/<tên-notebook>/outputs/*.jsonl {OUT}/      # khôi phục khi chạy tiếp
# !CUDA_VISIBLE_DEVICES=0 python src/infer_qwen.py --model {MODEL} --task extract  > /kaggle/working/log_full_extract.txt  2>&1 & \
#  CUDA_VISIBLE_DEVICES=1 python src/infer_qwen.py --model {MODEL} --task describe > /kaggle/working/log_full_describe.txt 2>&1 & wait
# !wc -l {OUT}/*.jsonl